# Text Index Generation

### This Notebook consists of:
 - Creation of a full text index in ElasticSearch for BM25 and Boolean Search
 - Bulk Insertions of the Dataset to the Index

In [ ]:
# Imports:
import os
import pandas as pd
from elasticsearch import Elasticsearch, helpers
import warnings 
import time
warnings.filterwarnings("ignore") # Turn off warnings
API_KEY = os.environ.get("ES_API_KEY")

In [ ]:
# Initialize Elasticsearch
client = Elasticsearch(
  "https://localhost:9200/",
  api_key=API_KEY,
  verify_certs=False
)

In [5]:
# Create the index -> Identical to BEIR Elasticsearch Index -> Use English Analyser
index_name = "dbpedia_v3"    
# Configurational Parameters:
mapping = {
    "mappings" : {
        "properties" : {
            "title": {"type": "text", "analyzer": "english"},
            "text": {"type": "text", "analyzer": "english"}
        }
    }
}
# Delete the index if it exists
client.indices.delete(index=index_name, ignore=[400, 404])
# Wait for deletion
time.sleep(2)
# Create new Index
client.indices.create(index=index_name, body=mapping, ignore=[400]) # 400 indicates existing index

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'dbpedia_v3'})

In [10]:
# Helper Function to create bulk actions for elasticsearch
def create_bulk_actions(data: pd.DataFrame) -> dict:
    """
    Create bulk actions for Elasticsearch
    params:
        data: pd.DataFrame
    returns:
        bulk_actions: list[dict]
    """
    bulk_actions = []
    for _, doc in data.iterrows():
        bulk_actions.append({
            "_id": str(doc["id"]),
            "_op_type": "index",
            "refresh": "wait_for",
            "text": doc["text"],
            "title": doc["title"],
            "dbpedia_id": str(doc["id"])
        })
    return bulk_actions

# Read the data from preprocessed
node_df = pd.read_parquet("../03_data/preprocessed_data/nodes_beir_dbpedia.parquet")
# Use helper function to insert bulk actions
helpers.bulk(client, create_bulk_actions(node_df), index=index_name)

(4635920, [])

## Add two new fields for boolean ranking

In [5]:
index_name = "dbpedia_v3"  
new_fields_mapping = {
    "properties": {
        "text_boolean_v2": {"type": "text", "similarity": "boolean"},
        "title_boolean_v2": {"type": "text", "similarity": "boolean"}
    }
}
client.indices.put_mapping(index=index_name, body=new_fields_mapping)

ObjectApiResponse({'acknowledged': True})

In [6]:
def update_in_batches(es: Elasticsearch, data:pd.DataFrame, batch_size: int):

    batch = []
    for _, doc in data.iterrows():
        # Update Operation for every doc
        update_action = {
            "_op_type": "update",
            "_index": index_name,
            "_id": str(doc["id"]),
            "doc": {
                "text_boolean_v2": doc["text"],
                "title_boolean_v2": doc["title"]
            }
        }
        batch.append(update_action)

        # Wenn die Batch-Größe erreicht ist, senden
        if len(batch) >= batch_size:
            helpers.bulk(es, batch)
            print(f"{len(batch)} Documents updated")
            batch.clear()  # Batch leeren

    # Rest in list
    if batch:
        helpers.bulk(es, batch)
        print(f"{len(batch)} Documents updated")

In [7]:
# Read the data from preprocessed
node_df = pd.read_parquet("../03_data/preprocessed_data/nodes_beir_dbpedia.parquet")

update_in_batches(client, node_df, 10000)

10000 Documents updated
10000 Documents updated
10000 Documents updated
10000 Documents updated
10000 Documents updated
10000 Documents updated
10000 Documents updated
10000 Documents updated
10000 Documents updated
10000 Documents updated
10000 Documents updated
10000 Documents updated
10000 Documents updated
10000 Documents updated
10000 Documents updated
10000 Documents updated
10000 Documents updated
10000 Documents updated
10000 Documents updated
10000 Documents updated
10000 Documents updated
10000 Documents updated
10000 Documents updated
10000 Documents updated
10000 Documents updated
10000 Documents updated
10000 Documents updated
10000 Documents updated
10000 Documents updated
10000 Documents updated
10000 Documents updated
10000 Documents updated
10000 Documents updated
10000 Documents updated
10000 Documents updated
10000 Documents updated
10000 Documents updated
10000 Documents updated
10000 Documents updated
10000 Documents updated
10000 Documents updated
10000 Documents 